# SSTW external baseline formal scoring Colab 入口

该 Notebook 单独运行现代视频水印 baseline adapter。它不会重新生成 Wan2.1 视频, 而是读取同一 workflow profile 的 run_root, 调用外部 baseline command, 写出 comparison records 和 evidence manifest。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import subprocess
import sys
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 1.1 可编辑 workflow profile 切换
# 修改 SSTW_WORKFLOW_PROFILE_VALUE 即可切换当前 Notebook 的运行层级。
# 留空表示使用当前 Notebook role 在统一配置中的默认 profile。
# 可选值: 'motion_calibration', 'validation_scale', 'pilot_paper'。
# 典型用法: motion_threshold_calibration_colab 使用 'motion_calibration'; 其他 3 个 Notebook 使用 'validation_scale' 或 'pilot_paper'。
import os

SSTW_WORKFLOW_PROFILE_VALUE = 'validation_scale'  # 当前显式运行 validation-scale

if SSTW_WORKFLOW_PROFILE_VALUE.strip():
    os.environ['SSTW_WORKFLOW_PROFILE'] = SSTW_WORKFLOW_PROFILE_VALUE.strip()
    print('SSTW_WORKFLOW_PROFILE =', os.environ['SSTW_WORKFLOW_PROFILE'])
else:
    os.environ.pop('SSTW_WORKFLOW_PROFILE', None)
    print('SSTW_WORKFLOW_PROFILE 未显式设置, 将使用当前 Notebook role 默认值。')


In [ ]:
# 2. 冷启动获取仓库代码
# Colab 环境不会保留代码, 每次运行都需要 clone 或 pull。
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 SSTW_REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"
!git rev-parse --short HEAD


In [ ]:
# 3. 安装 Colab GPU 运行依赖
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf huggingface_hub


In [ ]:
# 4. 可选 Hugging Face 认证
# token 只写入当前 Colab 环境变量, 不写入 records、tables、reports 或 package manifest。
try:
    from huggingface_hub import login
except Exception as exc:
    raise RuntimeError('huggingface_hub 未安装或无法导入, 请重新执行依赖安装单元') from exc

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 读取统一 workflow profile 配置并初始化 Drive 布局
from paper_workflow.notebook_utils import generative_video_model_probe_workflow as probe_workflow

NOTEBOOK_ROLE = 'external_baseline_formal_scoring'
WORKFLOW_PROFILE = os.environ.get('SSTW_WORKFLOW_PROFILE') or probe_workflow.default_workflow_profile_for_notebook_role(NOTEBOOK_ROLE)
workflow_profile = probe_workflow.resolve_notebook_workflow_profile(WORKFLOW_PROFILE, NOTEBOOK_ROLE)
PROFILE = workflow_profile['runtime_profile']
layout = probe_workflow.ensure_drive_layout(
    DRIVE_PROJECT_ROOT,
    workflow_profile=WORKFLOW_PROFILE,
    notebook_role=NOTEBOOK_ROLE,
)

print(json.dumps({
    'workflow_profile': workflow_profile,
    'layout': layout,
    'enabled_stage_plan': probe_workflow.build_workflow_stage_plan(WORKFLOW_PROFILE, NOTEBOOK_ROLE),
}, ensure_ascii=False, indent=2))


In [ ]:
# 6. 通用执行 helper
# Notebook 只调用仓库模块, 不在 cell 中手写正式 records、tables、figures 或 reports。
def stage_enabled(stage_name: str) -> bool:
    return probe_workflow.workflow_stage_enabled(WORKFLOW_PROFILE, NOTEBOOK_ROLE, stage_name)


def run_or_raise(stage_name: str, command: list[str]) -> None:
    print('\n===== stage:', stage_name, '=====')
    print(' '.join(command))
    result = probe_workflow.run_command(command)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit(result.returncode)


In [ ]:
# 8. 配置现代视频水印 baseline command
# 推荐通过 Colab Secrets 或前置 cell 写入 SSTW_<BASELINE>_EVAL_COMMAND 环境变量。
# 也可以在 BASELINE_COMMAND_TEMPLATES 中临时填写 baseline_id -> command_template。
BASELINE_COMMAND_TEMPLATES = {}
RUN_EXTERNAL_BASELINE_SOURCE_CLONE = os.environ.get('SSTW_RUN_EXTERNAL_BASELINE_SOURCE_CLONE', 'false').lower() == 'true'
REQUIRE_MODERN_BASELINE_COMMANDS_FOR_PAPER_GATE = os.environ.get('SSTW_REQUIRE_MODERN_BASELINE_COMMANDS_FOR_PAPER_GATE', 'true').lower() == 'true'
EXTERNAL_BASELINE_EVIDENCE_PATHS = [
    item for item in os.environ.get('SSTW_EXTERNAL_BASELINE_EVIDENCE_PATHS', '').split(os.pathsep)
    if item
]

modern_command_config_summary = probe_workflow.write_modern_baseline_colab_command_config_summary(
    layout,
    profile=WORKFLOW_PROFILE,
)
print(json.dumps(modern_command_config_summary, ensure_ascii=False, indent=2))

command_template_source = dict(os.environ)
command_template_source.update(BASELINE_COMMAND_TEMPLATES)
modern_command_env = probe_workflow.build_modern_baseline_command_env(PROFILE, command_template_source)
external_baseline_preflight_decision = probe_workflow.write_external_baseline_colab_preflight_decision(
    layout,
    profile=WORKFLOW_PROFILE,
    command_env=modern_command_env,
    require_modern_baseline_commands_for_paper_gate=REQUIRE_MODERN_BASELINE_COMMANDS_FOR_PAPER_GATE,
    run_external_baseline_source_clone=RUN_EXTERNAL_BASELINE_SOURCE_CLONE,
    evidence_paths=EXTERNAL_BASELINE_EVIDENCE_PATHS,
)
print(json.dumps(external_baseline_preflight_decision, ensure_ascii=False, indent=2))
probe_workflow.validate_modern_baseline_commands_for_profile(external_baseline_preflight_decision)

for env_name, command_template in modern_command_env.items():
    if command_template:
        os.environ[env_name] = command_template
os.environ['SSTW_EXTERNAL_BASELINE_EVIDENCE_PATHS'] = os.pathsep.join(EXTERNAL_BASELINE_EVIDENCE_PATHS)


In [ ]:
# 9. 执行 external_baseline source intake 与 adapter comparison
# 该 Notebook 不重新生成 Wan2.1 视频, 只读取同一 run_root 中已有的 runtime detection records。
if stage_enabled('external_baseline_source_intake'):
    run_or_raise(
        'external_baseline_source_intake',
        probe_workflow.build_external_baseline_source_intake_command(
            layout,
            execute_clone=RUN_EXTERNAL_BASELINE_SOURCE_CLONE,
        ),
    )

if stage_enabled('external_baseline_comparison'):
    run_or_raise('external_baseline_comparison', probe_workflow.build_external_baseline_comparison_command(layout))


In [ ]:
# 末尾治理审计与 Google Drive 打包
INCLUDE_VIDEOS_IN_PACKAGE = os.environ.get('SSTW_INCLUDE_VIDEOS_IN_PACKAGE', 'true').lower() == 'true'

if stage_enabled('quick_tests_and_harness'):
    !pytest -q
    !python tools/harness/run_all_audits.py

if stage_enabled('drive_packaging'):
    run_or_raise('drive_packaging', probe_workflow.build_drive_packaging_command(layout, include_videos=INCLUDE_VIDEOS_IN_PACKAGE))

package_dir = Path(layout['drive_package_dir'])
print('package_dir =', package_dir)
if package_dir.exists():
    for path in sorted(package_dir.glob('*')):
        print(path)
